# Direct multi-horizon visitor forecasting

This notebook forecasts every `PLACEKEY × date` in 2025 directly from a
single production origin, **2024-12-31**. A predicted day is never used as
an input to another predicted day.

The historical test follows this structure:

- Train/tune on earlier fixed-origin yearly forecasts.
- Lock the method.
- At origin **2023-12-31**, predict all of 2024 without using any 2024
  observations as features.
- Refit through **2024-12-31** and independently predict all of 2025.

This is a pooled direct model: one LightGBM model learns all horizons.

In [1]:
from pathlib import Path
import json
import gc

import lightgbm as lgb
import numpy as np
import pandas as pd
from IPython.display import display
from pandas.tseries.holiday import USFederalHolidayCalendar


PARQUET_PATH = Path(
    "/Users/rusli/Documents/GitHub/Rice-To-Meet-You/"
    "visitor_data/output/store_visits_core_poi_spend_12m_updated.parquet"
)
OUTPUT_PATH = Path(
    "/Users/rusli/Documents/GitHub/Rice-To-Meet-You/"
    "visitor_data/output/visitor_predictions_2025_direct.csv"
)

TARGET = "AVERAGE_DAILY_VISITS"
FORECAST_ORIGIN = pd.Timestamp("2024-12-31")
FORECAST_DATES = pd.date_range("2025-01-01", "2025-12-31", freq="D")

STATIC_CATEGORICAL = [
    "BRAND", "NAICS_CODE_2022", "SUB_CATEGORY_2022", "TOP_CATEGORY_2022"
]
RAW_COLUMNS = [
    "PLACEKEY", "LOCAL_DATE", TARGET,
    *STATIC_CATEGORICAL, "CATEGORY_TAGS",
]

if not PARQUET_PATH.exists():
    raise FileNotFoundError(PARQUET_PATH)

print("Reading only the columns needed by the direct model...")
daily = pd.read_parquet(PARQUET_PATH, columns=RAW_COLUMNS)
daily["LOCAL_DATE"] = pd.to_datetime(daily["LOCAL_DATE"])
daily = daily.loc[daily["LOCAL_DATE"] <= FORECAST_ORIGIN].copy()
daily = daily.sort_values(["PLACEKEY", "LOCAL_DATE"]).reset_index(drop=True)

if daily.duplicated(["PLACEKEY", "LOCAL_DATE"]).any():
    raise ValueError("PLACEKEY + LOCAL_DATE is not unique")
if daily[TARGET].isna().any():
    raise ValueError(f"{TARGET} contains missing values")

print(
    f"Loaded {len(daily):,} rows, {daily['PLACEKEY'].nunique():,} PLACEKEYs, "
    f"{daily['LOCAL_DATE'].min().date()} through {daily['LOCAL_DATE'].max().date()}"
)

Reading only the columns needed by the direct model...


Loaded 6,858,558 rows, 3,754 PLACEKEYs, 2020-01-01 through 2024-12-31


## Exclude PLACEKEYs with an implausible visit scale (data quality)

A subset of PLACEKEYs carry `AVERAGE_DAILY_VISITS` values in the tens/hundreds of thousands per
day — physically impossible for a single store. Root cause (see
`VisitorTimeSeriesPrediction.ipynb` Step 1.5): for `MATCH_STATUS == "ambiguous_brand_market_match"`
rows, `AVERAGE_DAILY_VISITS == TOTAL_DAILY_VISITS / CORE_POI_MATCH_COUNT` — a brand-wide market
visit total divided evenly across however many candidate POIs SafeGraph could find, then written
onto *each* one as if it were that store's own average.

A direct `MATCH_STATUS` filter would remove that value's synthetic-count source at the root, but
it also removes 72% of PLACEKEYs (2,713 of 3,754) — most ambiguous-match stores' divided values
still look plausible, just wrong for the wrong reason. A sanity cap on scale instead catches only
the worst offenders, trading completeness for keeping most of the panel — same threshold as Step
1.5, so results stay comparable across both notebooks.

In [2]:
IMPLAUSIBLE_VISITS_THRESHOLD = 20_000

mean_visits_by_placekey = daily.groupby("PLACEKEY")[TARGET].mean()
implausible_placekeys = mean_visits_by_placekey[
    mean_visits_by_placekey > IMPLAUSIBLE_VISITS_THRESHOLD
].index.tolist()

daily = daily.loc[~daily["PLACEKEY"].isin(implausible_placekeys)].reset_index(drop=True)

print(
    f"Excluding {len(implausible_placekeys):,} of {mean_visits_by_placekey.shape[0]:,} PLACEKEYs "
    f"({len(implausible_placekeys) / mean_visits_by_placekey.shape[0]:.2%}) with mean "
    f"{TARGET} > {IMPLAUSIBLE_VISITS_THRESHOLD:,}"
)
print(f"Remaining: {len(daily):,} rows, {daily['PLACEKEY'].nunique():,} PLACEKEYs")

Excluding 272 of 3,754 PLACEKEYs (7.25%) with mean AVERAGE_DAILY_VISITS > 20,000
Remaining: 6,361,614 rows, 3,482 PLACEKEYs


## Historical features that remain valid at a fixed origin

A future target date may use its same calendar date one, two, or three years
earlier because all those observations exist by the year-end origin. Rolling
summaries are attached to those historical reference dates—not calculated
from observations inside the forecast year.

Calendar-year offsets are used instead of row-based `shift(365)`, avoiding
the leap-year one-day error.

In [3]:
visits = daily.groupby("PLACEKEY", sort=False)[TARGET]
daily["history_roll_7"] = visits.transform(
    lambda s: s.rolling(7, min_periods=1).mean()
)
daily["history_roll_28"] = visits.transform(
    lambda s: s.rolling(28, min_periods=1).mean()
)
daily["history_roll_365"] = visits.transform(
    lambda s: s.rolling(365, min_periods=30).mean()
)

def category_tag_count(raw) -> float:
    if not isinstance(raw, str):
        return 0.0
    try:
        value = json.loads(raw)
        return float(len(value)) if isinstance(value, list) else 0.0
    except (TypeError, ValueError):
        return 0.0


daily["category_tag_count"] = daily["CATEGORY_TAGS"].map(category_tag_count)

In [4]:
def add_target_calendar_features(frame: pd.DataFrame) -> pd.DataFrame:
    d = pd.to_datetime(frame["TARGET_DATE"])
    day_of_year = d.dt.dayofyear.astype(float)
    frame["target_month"] = d.dt.month.astype(float)
    frame["target_day_of_week"] = d.dt.dayofweek.astype(float)
    frame["target_day_of_month"] = d.dt.day.astype(float)
    frame["target_week_of_year"] = d.dt.isocalendar().week.astype(float)
    frame["target_is_weekend"] = (d.dt.dayofweek >= 5).astype(float)
    frame["target_doy_sin"] = np.sin(2 * np.pi * day_of_year / 365.25)
    frame["target_doy_cos"] = np.cos(2 * np.pi * day_of_year / 365.25)

    holidays = USFederalHolidayCalendar().holidays(
        start=d.min() - pd.Timedelta(days=7),
        end=d.max() + pd.Timedelta(days=7),
    )
    frame["target_is_federal_holiday"] = d.isin(holidays).astype(float)
    frame["target_is_thanksgiving"] = (
        (d.dt.month == 11)
        & (d.dt.dayofweek == 3)
        & (((d.dt.day - 1) // 7 + 1) == 4)
    ).astype(float)
    frame["target_is_black_friday"] = (
        (d.dt.month == 11)
        & (d.dt.dayofweek == 4)
        & (((d.dt.day - 1) // 7 + 1) == 4)
    ).astype(float)
    frame["target_is_christmas"] = (
        (d.dt.month == 12) & (d.dt.day == 25)
    ).astype(float)
    return frame


def calendar_years_before(dates: pd.Series, years: int) -> pd.Series:
    # DateOffset maps leap day to February 28 in a non-leap reference year.
    return dates - pd.DateOffset(years=years)

## Build one leakage-safe direct forecast frame

For an origin such as `2023-12-31`, every feature reference is asserted to
be on or before that origin. Historical frames contain the future target
only as the label; the label is never included in `FEATURE_COLUMNS`.

In [5]:
HISTORY_LOOKUP_COLUMNS = [
    "PLACEKEY", "LOCAL_DATE", TARGET,
    "history_roll_7", "history_roll_28", "history_roll_365",
]

def make_direct_frame(
    origin,
    target_dates=None,
    include_target: bool = True,
) -> pd.DataFrame:
    origin = pd.Timestamp(origin)
    start = origin + pd.Timedelta(days=1)
    end = origin + pd.DateOffset(years=1)

    origin_rows = daily.loc[daily["LOCAL_DATE"] == origin].copy()
    if origin_rows.empty:
        raise ValueError(f"No observations exist at origin {origin.date()}")

    if include_target:
        base = daily.loc[
            daily["LOCAL_DATE"].between(start, end),
            ["PLACEKEY", "LOCAL_DATE", TARGET],
        ].rename(columns={"LOCAL_DATE": "TARGET_DATE", TARGET: "y"})
        base = base.loc[base["PLACEKEY"].isin(origin_rows["PLACEKEY"])].copy()
    else:
        dates = pd.DatetimeIndex(target_dates)
        if dates.min() < start or dates.max() > end:
            raise ValueError("Target dates must be within the year after origin")
        index = pd.MultiIndex.from_product(
            [origin_rows["PLACEKEY"].unique(), dates],
            names=["PLACEKEY", "TARGET_DATE"],
        )
        base = index.to_frame(index=False)

    base["ORIGIN_DATE"] = origin
    base["horizon_days"] = (
        base["TARGET_DATE"] - origin
    ).dt.days.astype(float)
    base["origin_year"] = float(origin.year)
    base = add_target_calendar_features(base)

    # lag364_baseline -- 364 days back (52 weeks), preserves day-of-week
    # unlike seasonal_lag_1y's calendar-date match. Retail visits are far
    # more weekday-dependent than date-dependent, so this is typically a
    # stronger naive baseline to compare the model against.
    base["_lag364_date"] = base["TARGET_DATE"] - pd.Timedelta(days=364)
    lag364_lookup = daily[["PLACEKEY", "LOCAL_DATE", TARGET]].rename(
        columns={"LOCAL_DATE": "_lag364_date", TARGET: "lag364_baseline"}
    )
    base = base.merge(lag364_lookup, on=["PLACEKEY", "_lag364_date"], how="left")
    base = base.drop(columns=["_lag364_date"])

    history_lookup = daily[HISTORY_LOOKUP_COLUMNS]
    for years_back in (1, 2, 3):
        ref_col = f"reference_date_{years_back}y"
        base[ref_col] = calendar_years_before(base["TARGET_DATE"], years_back)
        if (base[ref_col] > origin).any():
            raise AssertionError(
                f"{ref_col} crossed forecast origin {origin.date()}"
            )

        renamed = history_lookup.rename(columns={
            "LOCAL_DATE": ref_col,
            TARGET: f"seasonal_lag_{years_back}y",
            "history_roll_7": f"seasonal_roll_7_{years_back}y",
            "history_roll_28": f"seasonal_roll_28_{years_back}y",
            "history_roll_365": f"seasonal_roll_365_{years_back}y",
        })
        base = base.merge(renamed, on=["PLACEKEY", ref_col], how="left")

    origin_keep = [
        "PLACEKEY", TARGET, "history_roll_7", "history_roll_28",
        "history_roll_365", *STATIC_CATEGORICAL, "category_tag_count",
    ]
    origin_features = origin_rows[origin_keep].rename(columns={
        TARGET: "origin_visits",
        "history_roll_7": "origin_roll_7",
        "history_roll_28": "origin_roll_28",
        "history_roll_365": "origin_roll_365",
    })
    base = base.merge(origin_features, on="PLACEKEY", how="left")

    base["seasonal_change_1y"] = (
        base["seasonal_lag_1y"] - base["seasonal_lag_2y"]
    )
    base["seasonal_change_2y"] = (
        base["seasonal_lag_2y"] - base["seasonal_lag_3y"]
    )
    base["seasonal_ratio_1y"] = (
        (base["seasonal_lag_1y"] + 1.0)
        / (base["seasonal_lag_2y"] + 1.0)
    )

    # These columns are diagnostic only and are excluded from the model.
    reference_columns = [
        f"reference_date_{years_back}y" for years_back in (1, 2, 3)
    ]
    assert all((base[c] <= origin).all() for c in reference_columns)
    return base

In [6]:
CATEGORICAL_COLUMNS = ["PLACEKEY", *STATIC_CATEGORICAL]
NUMERIC_COLUMNS = [
    "horizon_days", "origin_year",
    "target_month", "target_day_of_week", "target_day_of_month",
    "target_week_of_year", "target_is_weekend",
    "target_doy_sin", "target_doy_cos",
    "target_is_federal_holiday", "target_is_thanksgiving",
    "target_is_black_friday", "target_is_christmas",
    "origin_visits", "origin_roll_7", "origin_roll_28", "origin_roll_365",
    "seasonal_lag_1y", "seasonal_lag_2y", "seasonal_lag_3y",
    "seasonal_roll_7_1y", "seasonal_roll_28_1y", "seasonal_roll_365_1y",
    "seasonal_roll_7_2y", "seasonal_roll_28_2y", "seasonal_roll_365_2y",
    "seasonal_change_1y", "seasonal_change_2y", "seasonal_ratio_1y",
    "category_tag_count",
]
FEATURE_COLUMNS = CATEGORICAL_COLUMNS + NUMERIC_COLUMNS


def model_frames(train: pd.DataFrame, test: pd.DataFrame):
    X_train = train[FEATURE_COLUMNS].copy()
    X_test = test[FEATURE_COLUMNS].copy()
    categories = {}
    for col in CATEGORICAL_COLUMNS:
        cats = pd.Categorical(X_train[col].astype("string")).categories
        categories[col] = cats
        X_train[col] = pd.Categorical(
            X_train[col].astype("string"), categories=cats
        )
        X_test[col] = pd.Categorical(
            X_test[col].astype("string"), categories=cats
        )
    return X_train, X_test, categories


def direct_model():
    return lgb.LGBMRegressor(
        objective="regression_l1",
        n_estimators=650,
        learning_rate=0.04,
        num_leaves=63,
        min_child_samples=100,
        subsample=0.85,
        colsample_bytree=0.85,
        random_state=0,
        verbosity=-1,
        n_jobs=-1,
    )


def score(actual, predicted) -> dict:
    actual = np.asarray(actual, dtype=float)
    predicted = np.clip(np.asarray(predicted, dtype=float), 0, None)
    residual = actual - predicted
    denominator = np.abs(actual) + np.abs(predicted)
    smape_terms = np.divide(
        2 * np.abs(residual), denominator,
        out=np.zeros_like(denominator), where=denominator != 0,
    )
    ss_res = float(np.sum(residual ** 2))
    ss_tot = float(np.sum((actual - actual.mean()) ** 2))
    return {
        "MAE": float(np.mean(np.abs(residual))),
        "RMSE": float(np.sqrt(np.mean(residual ** 2))),
        "sMAPE": float(np.mean(smape_terms) * 100),
        "R2": 1 - ss_res / ss_tot,
    }


def fit_and_predict(train: pd.DataFrame, test: pd.DataFrame):
    X_train, X_test, categories = model_frames(train, test)
    model = direct_model()
    model.fit(X_train, np.log1p(train["y"].to_numpy(dtype=float)))
    prediction = np.clip(np.expm1(model.predict(X_test)), 0, None)
    return model, prediction, categories

## Recursive and hybrid legs (comparison only)

Two more strategies are scored alongside the direct model, so its
MAE has more to beat than just the naive baseline:

- **Recursive** — a single one-step-ahead LightGBM model, rolled forward one day at a time for
  the full target year. Each day's `visits_lag_1/7/28` and rolling means come from real history
  *or the model's own prior predictions in this same rollout*, so its errors can compound over a
  long horizon — the opposite of the direct model's guarantee.
- **Hybrid** — recursive for `horizon_days <= HYBRID_THRESHOLD_DAYS`, then the direct model above for everything past
  it. `HYBRID_THRESHOLD_DAYS = 90` follows this repo's own prior sweep in
  `VisitorTimeSeriesPrediction.ipynb` (Step 9), where a 90-day threshold beat both a 28-day
  threshold and pure direct for every model tried, LightGBM included.

Only the direct model feeds the production 2025 forecast further down — these two are
diagnostic, scored at the 2023 and 2024 origins alongside it.

In [7]:
RECURSIVE_LAGS = (1, 7, 28)
RECURSIVE_ROLLING_WINDOWS = (7, 28)
HYBRID_THRESHOLD_DAYS = 90

recursive_visits = daily.groupby("PLACEKEY", sort=False)[TARGET]
for lag in RECURSIVE_LAGS:
    daily[f"visits_lag_{lag}"] = recursive_visits.shift(lag)
for window in RECURSIVE_ROLLING_WINDOWS:
    daily[f"visits_roll_{window}"] = recursive_visits.transform(
        lambda s, size=window: s.shift(1).rolling(size, min_periods=1).mean()
    )

_recursive_calendar = add_target_calendar_features(
    pd.DataFrame({"TARGET_DATE": daily["LOCAL_DATE"]})
)
RECURSIVE_CALENDAR_COLUMNS = [
    c for c in _recursive_calendar.columns if c != "TARGET_DATE"
]
daily[RECURSIVE_CALENDAR_COLUMNS] = _recursive_calendar[RECURSIVE_CALENDAR_COLUMNS]

RECURSIVE_CATEGORICAL_COLUMNS = CATEGORICAL_COLUMNS
RECURSIVE_NUMERIC_COLUMNS = [
    *RECURSIVE_CALENDAR_COLUMNS,
    *[f"visits_lag_{lag}" for lag in RECURSIVE_LAGS],
    *[f"visits_roll_{w}" for w in RECURSIVE_ROLLING_WINDOWS],
    "category_tag_count",
]
RECURSIVE_FEATURE_COLUMNS = RECURSIVE_CATEGORICAL_COLUMNS + RECURSIVE_NUMERIC_COLUMNS


def recursive_model():
    return lgb.LGBMRegressor(
        objective="regression_l1",
        n_estimators=650,
        learning_rate=0.04,
        num_leaves=63,
        min_child_samples=100,
        subsample=0.85,
        colsample_bytree=0.85,
        random_state=0,
        verbosity=-1,
        n_jobs=-1,
    )


def fit_recursive_model(train_end):
    train_rows = daily.loc[
        daily["LOCAL_DATE"] <= pd.Timestamp(train_end)
    ].dropna(subset=RECURSIVE_NUMERIC_COLUMNS)
    X_train = train_rows[RECURSIVE_FEATURE_COLUMNS].copy()
    categories = {}
    for col in RECURSIVE_CATEGORICAL_COLUMNS:
        cats = pd.Categorical(X_train[col].astype("string")).categories
        categories[col] = cats
        X_train[col] = pd.Categorical(X_train[col].astype("string"), categories=cats)
    model = recursive_model()
    model.fit(X_train, np.log1p(train_rows[TARGET].to_numpy(dtype=float)))
    return model, categories


def recursive_placekey_forecast(origin, forecast_dates, model, categories) -> pd.DataFrame:
    """Day-by-day rollout: each date's lag/rolling features come from real
    history, plus every earlier date's own prediction in this same loop."""
    origin = pd.Timestamp(origin)
    forecast_dates = pd.DatetimeIndex(forecast_dates)
    history = daily.loc[daily["LOCAL_DATE"] <= origin, ["PLACEKEY", "LOCAL_DATE", TARGET]]
    static_lookup = (
        daily.loc[daily["LOCAL_DATE"] == origin]
        .set_index("PLACEKEY")[[*STATIC_CATEGORICAL, "category_tag_count"]]
        .to_dict(orient="index")
    )
    calendar_lookup = add_target_calendar_features(
        pd.DataFrame({"TARGET_DATE": forecast_dates})
    ).set_index("TARGET_DATE")

    max_lag = max(RECURSIVE_LAGS)
    series = {
        placekey: group.sort_values("LOCAL_DATE")[TARGET].astype(float).tolist()
        for placekey, group in history.groupby("PLACEKEY")
    }

    predictions = []
    for forecast_date in forecast_dates:
        calendar_row = calendar_lookup.loc[forecast_date].to_dict()
        rows, placekeys = [], []
        for placekey, values in series.items():
            if len(values) < max_lag:
                continue
            rows.append({
                "PLACEKEY": placekey,
                **static_lookup.get(placekey, {}),
                "visits_lag_1": values[-1],
                "visits_lag_7": values[-7],
                "visits_lag_28": values[-28],
                "visits_roll_7": float(np.mean(values[-7:])),
                "visits_roll_28": float(np.mean(values[-28:])),
                **calendar_row,
            })
            placekeys.append(placekey)

        features = pd.DataFrame(rows)
        for col in RECURSIVE_CATEGORICAL_COLUMNS:
            features[col] = pd.Categorical(
                features[col].astype("string"), categories=categories[col]
            )
        predicted = np.clip(np.expm1(model.predict(features[RECURSIVE_FEATURE_COLUMNS])), 0, None)

        for placekey, value in zip(placekeys, predicted):
            value = float(value)
            series[placekey].append(value)
            predictions.append({
                "PLACEKEY": placekey, "TARGET_DATE": forecast_date, "recursive_pred": value,
            })
    return pd.DataFrame(predictions)


def hybrid_predictions(
    frame: pd.DataFrame, direct_pred: np.ndarray, recursive_df: pd.DataFrame,
    threshold_days: int = HYBRID_THRESHOLD_DAYS,
) -> np.ndarray:
    """h <= threshold_days uses the recursive rollout; h > threshold_days uses
    the already-fitted direct model's prediction at the same origin."""
    merged = frame[["PLACEKEY", "TARGET_DATE", "horizon_days"]].copy()
    merged["direct_pred"] = direct_pred
    merged = merged.merge(recursive_df, on=["PLACEKEY", "TARGET_DATE"], how="left")
    use_recursive = (merged["horizon_days"] <= threshold_days) & merged["recursive_pred"].notna()
    return np.where(use_recursive, merged["recursive_pred"], merged["direct_pred"])

## Validation and final untouched test

The configuration is inspected on the 2023 fixed-origin forecast. After it
is locked, the 2024 forecast is generated once from `2023-12-31`. No actual
2024 lag or rolling value can enter the 2024 feature matrix.

The recursive and hybrid legs defined above are fit and scored at both
origins too, so the direct model's MAE has something to beat besides the
naive `lag364` baseline. Only the direct model feeds the production 2025
forecast below.

In [8]:
print("Building direct training frames...")
frame_2021 = make_direct_frame("2020-12-31")
frame_2022 = make_direct_frame("2021-12-31")
frame_2023 = make_direct_frame("2022-12-31")

validation_train = pd.concat([frame_2021, frame_2022], ignore_index=True)
validation_model, validation_pred, _ = fit_and_predict(
    validation_train, frame_2023
)

print("Fitting recursive model and rolling it forward through 2023...")
validation_recursive_model, validation_recursive_categories = fit_recursive_model("2022-12-31")
validation_recursive_df = recursive_placekey_forecast(
    "2022-12-31", pd.date_range("2023-01-01", "2023-12-31", freq="D"),
    validation_recursive_model, validation_recursive_categories,
)
validation_recursive_eval = frame_2023.merge(
    validation_recursive_df, on=["PLACEKEY", "TARGET_DATE"], how="inner"
)
validation_hybrid_pred = hybrid_predictions(frame_2023, validation_pred, validation_recursive_df)

validation_results = pd.DataFrame([
    {
        "split": "2023 pooled-direct model",
        **score(frame_2023["y"], validation_pred),
    },
    {
        "split": "2023 recursive model",
        **score(validation_recursive_eval["y"], validation_recursive_eval["recursive_pred"]),
    },
    {
        "split": f"2023 hybrid model (recursive <= {HYBRID_THRESHOLD_DAYS}d, direct after)",
        **score(frame_2023["y"], validation_hybrid_pred),
    },
    {
        "split": "2023 lag364 baseline",
        **score(frame_2023["y"], frame_2023["lag364_baseline"]),
    },
])
display(validation_results)

del validation_model, validation_train, validation_pred
del validation_recursive_model, validation_recursive_df, validation_recursive_eval, validation_hybrid_pred
gc.collect()

Building direct training frames...


Fitting recursive model and rolling it forward through 2023...


,split,MAE,RMSE,sMAPE,R2
0,2023 pooled-direct model,330.984846,826.131144,19.546053,0.967988
1,2023 recursive model,480.313624,1131.154583,31.738302,0.939985
2,"2023 hybrid model (recursive <= 90d, direct af...",345.422009,850.930331,20.639372,0.966037
3,2023 lag364 baseline,310.004136,859.113370,13.413659,0.965381


0

## Feature-group ablation (leave-one-group-out)

Every column in `FEATURE_COLUMNS` above is already part of the production feature set — unlike
`VisitorTimeSeriesPrediction.ipynb`'s Step 2.5, there is no held-out pool of candidate columns to
greedily add one at a time. So this version removes one feature group at a time from the full
set instead, refits on the same 2021–2022 train / 2023 validation split used just above, and
compares each reduced model's MAE to the full model's `full_val_mae`. A group is **load-bearing**
if dropping it costs a **≥1% relative MAE increase**.

`PLACEKEY`, `horizon_days`, and `origin_year` are excluded from ablation — they are structural to
a pooled, per-store, multi-horizon direct model, not optional signal to test.

This is diagnostic only: it does not modify `FEATURE_COLUMNS`, and it stays on the untouched-2024
side of this notebook's existing discipline — only the 2023 validation split (already computed
above) is used here.

In [9]:
ABLATION_ACCEPT_THRESHOLD = 0.01  # >=1% relative MAE increase on removal = load-bearing

FEATURE_GROUPS = {
    "target_calendar": [
        "target_month", "target_day_of_week", "target_day_of_month",
        "target_week_of_year", "target_is_weekend",
        "target_doy_sin", "target_doy_cos",
        "target_is_federal_holiday", "target_is_thanksgiving",
        "target_is_black_friday", "target_is_christmas",
    ],
    "origin_snapshot": [
        "origin_visits", "origin_roll_7", "origin_roll_28", "origin_roll_365",
    ],
    "seasonal_lags": [
        "seasonal_lag_1y", "seasonal_lag_2y", "seasonal_lag_3y",
        "seasonal_change_1y", "seasonal_change_2y", "seasonal_ratio_1y",
    ],
    "seasonal_rolling": [
        "seasonal_roll_7_1y", "seasonal_roll_28_1y", "seasonal_roll_365_1y",
        "seasonal_roll_7_2y", "seasonal_roll_28_2y", "seasonal_roll_365_2y",
    ],
    "business_identity": [
        "BRAND", "NAICS_CODE_2022", "SUB_CATEGORY_2022", "TOP_CATEGORY_2022",
        "category_tag_count",
    ],
}


def _ablation_fit_predict_mae(train, test, feature_columns, categorical_columns) -> float:
    X_train = train[feature_columns].copy()
    X_test = test[feature_columns].copy()
    for col in categorical_columns:
        cats = pd.Categorical(X_train[col].astype("string")).categories
        X_train[col] = pd.Categorical(X_train[col].astype("string"), categories=cats)
        X_test[col] = pd.Categorical(X_test[col].astype("string"), categories=cats)
    model = direct_model()
    model.fit(X_train, np.log1p(train["y"].to_numpy(dtype=float)))
    prediction = np.clip(np.expm1(model.predict(X_test)), 0, None)
    return score(test["y"], prediction)["MAE"]


ablation_train = pd.concat([frame_2021, frame_2022], ignore_index=True)
full_val_mae = validation_results.loc[
    validation_results["split"] == "2023 pooled-direct model", "MAE"
].item()

ablation_rows = [
    {"group": "full feature set", "val_MAE": full_val_mae, "delta_pct": 0.0, "load_bearing": None}
]
for group_name, group_columns in FEATURE_GROUPS.items():
    reduced_categorical = [c for c in CATEGORICAL_COLUMNS if c not in group_columns]
    reduced_numeric = [c for c in NUMERIC_COLUMNS if c not in group_columns]
    mae = _ablation_fit_predict_mae(
        ablation_train, frame_2023, reduced_categorical + reduced_numeric, reduced_categorical
    )
    delta_pct = (mae - full_val_mae) / full_val_mae * 100
    load_bearing = delta_pct >= ABLATION_ACCEPT_THRESHOLD * 100
    print(
        f"drop {group_name}: val_MAE={mae:.4f}  delta={delta_pct:+.2f}%  "
        f"-> {'load-bearing' if load_bearing else 'not load-bearing'}"
    )
    ablation_rows.append({
        "group": f"- {group_name}", "val_MAE": mae, "delta_pct": delta_pct,
        "load_bearing": load_bearing,
    })

ablation_summary = pd.DataFrame(ablation_rows)
display(ablation_summary)

del ablation_train
gc.collect()

drop target_calendar: val_MAE=473.6568  delta=+43.11%  -> load-bearing


drop origin_snapshot: val_MAE=338.3809  delta=+2.23%  -> load-bearing


drop seasonal_lags: val_MAE=335.1997  delta=+1.27%  -> load-bearing


drop seasonal_rolling: val_MAE=341.9835  delta=+3.32%  -> load-bearing


drop business_identity: val_MAE=353.2954  delta=+6.74%  -> load-bearing


,group,val_MAE,delta_pct,load_bearing
0,full feature set,330.984846,0.000000,None
1,- target_calendar,473.656795,43.105281,True
2,- origin_snapshot,338.380854,2.234546,True
3,- seasonal_lags,335.199654,1.273414,True
4,- seasonal_rolling,341.983536,3.323019,True
5,- business_identity,353.295400,6.740657,True


166

## Feature-group ablation (forward add-one-in — new feature groups)

The leave-one-out pass above only tells us whether the columns already in `FEATURE_COLUMNS` are
worth keeping. This second pass runs the ablation the other direction, mirroring
`VisitorTimeSeriesPrediction.ipynb` Step 2.5's original design: introduce a **candidate** feature
group that isn't in the model yet, one at a time, on top of the full current feature set, and
keep it only if it earns a **≥1% relative MAE reduction** on the same 2021–2022 train / 2023
validation split.

`business_identity` (`BRAND`) and `business_type` (`NAICS_CODE_2022`/`SUB_CATEGORY_2022`/
`TOP_CATEGORY_2022`/`category_tag_count`) from Step 2.5's candidate pool are skipped — both are
already in this notebook's baseline. `rolling_extended` is also skipped: unlike Step 2.5's
recursive-style model, a direct forecast would need each new rolling stat re-attached at the
origin *and* at all three seasonal reference dates to stay leakage-safe, roughly quadrupling this
section's size for signal that likely overlaps heavily with the existing `roll_7/28/365` means
already in the model.

Remaining candidates: `geography`, `physical_poi`, `location_structure`, `operating_status`,
`calendar_extended` (only the pieces not already covered by this notebook's own target-calendar
features), `spatial_neighborhood`, and `historical_spending`. `historical_spending` is attached at
each row's own `ORIGIN_DATE` rather than `TARGET_DATE` — a direct forecast can only condition on
spend behavior already known at the forecast origin, and the leakage check in
`VisitorTimeSeriesPrediction.ipynb` Step 2.5 (referenced in the data-quality cell above) already
established that a `LOCAL_DATE` row's stamped spend value never looks past that same date.

In [10]:
EXTRA_RAW_COLUMNS = [
    "PLACEKEY", "LOCAL_DATE",
    "MARKET_VISIT", "REGION", "LATITUDE", "LONGITUDE",
    "WKT_AREA_SQ_METERS", "POLYGON_CLASS", "ENCLOSED", "INCLUDES_PARKING_LOT",
    "OPENED_ON", "CLOSED_ON", "OPEN_HOURS", "PARENT_PLACEKEY", "IS_SYNTHETIC",
    "SPEND_MEDIAN_SPEND_PER_CUSTOMER", "SPEND_MEDIAN_SPEND_PER_TRANSACTION",
    "SPEND_ONLINE_SPEND", "SPEND_ONLINE_TRANSACTIONS",
    "SPEND_RAW_NUM_CUSTOMERS", "SPEND_RAW_NUM_TRANSACTIONS", "SPEND_RAW_TOTAL_SPEND",
    "SPEND_PCT_CHANGE_VS_PREV_MONTH", "SPEND_PCT_CHANGE_VS_PREV_YEAR",
]
SPEND_NUMERIC_COLUMNS = [c for c in EXTRA_RAW_COLUMNS if c.startswith("SPEND_")]

_extra_raw = pd.read_parquet(PARQUET_PATH, columns=EXTRA_RAW_COLUMNS)
_extra_raw["LOCAL_DATE"] = pd.to_datetime(_extra_raw["LOCAL_DATE"])
_extra_raw = _extra_raw.loc[_extra_raw["PLACEKEY"].isin(daily["PLACEKEY"].unique())]

candidate_static = _extra_raw.drop_duplicates(subset="PLACEKEY").set_index("PLACEKEY")
spend_by_key = _extra_raw[["PLACEKEY", "LOCAL_DATE", *SPEND_NUMERIC_COLUMNS]].rename(
    columns={"LOCAL_DATE": "ORIGIN_DATE"}
)


def _static_map(frame, column):
    return frame["PLACEKEY"].map(candidate_static[column])


def group_geography(frame):
    frame["MARKET_VISIT"] = _static_map(frame, "MARKET_VISIT")
    frame["REGION"] = _static_map(frame, "REGION")
    frame["cand_latitude"] = _static_map(frame, "LATITUDE").astype(float)
    frame["cand_longitude"] = _static_map(frame, "LONGITUDE").astype(float)
    return ["cand_latitude", "cand_longitude"], ["MARKET_VISIT", "REGION"]
    # CITY / MARKET_CORE omitted -- identical to MARKET_VISIT repo-wide, see
    # VisitorTimeSeriesPrediction.ipynb Step 2.5.


def group_physical_poi(frame):
    frame["cand_area_sq_meters"] = _static_map(frame, "WKT_AREA_SQ_METERS").astype(float)
    frame["cand_enclosed"] = _static_map(frame, "ENCLOSED").astype(float)
    frame["cand_includes_parking_lot"] = _static_map(frame, "INCLUDES_PARKING_LOT").astype(float)
    frame["POLYGON_CLASS"] = _static_map(frame, "POLYGON_CLASS")
    return ["cand_area_sq_meters", "cand_enclosed", "cand_includes_parking_lot"], ["POLYGON_CLASS"]


def group_location_structure(frame):
    frame["cand_has_parent_placekey"] = _static_map(frame, "PARENT_PLACEKEY").notna().astype(float)
    frame["cand_is_synthetic"] = _static_map(frame, "IS_SYNTHETIC").astype(float)
    return ["cand_has_parent_placekey", "cand_is_synthetic"], []


def _parse_open_hours_per_week(raw):
    if not isinstance(raw, str):
        return np.nan
    try:
        spec = json.loads(raw)
    except (ValueError, TypeError):
        return np.nan
    total_minutes = 0.0
    for ranges in spec.values():
        for start, end in ranges:
            sh, sm = map(int, start.split(":"))
            eh, em = map(int, end.split(":"))
            total_minutes += max(0, (eh * 60 + em) - (sh * 60 + sm))
    return total_minutes / 60.0


_DOW_KEYS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]


def _parse_open_today_lookup(raw):
    if not isinstance(raw, str):
        return {i: np.nan for i in range(7)}
    try:
        spec = json.loads(raw)
    except (ValueError, TypeError):
        return {i: np.nan for i in range(7)}
    return {i: float(len(spec.get(k, [])) > 0) for i, k in enumerate(_DOW_KEYS)}


_open_hours_per_week = candidate_static["OPEN_HOURS"].apply(_parse_open_hours_per_week)
_open_today_lookup = candidate_static["OPEN_HOURS"].apply(_parse_open_today_lookup)


def group_operating_status(frame):
    opened = frame["PLACEKEY"].map(candidate_static["OPENED_ON"]).pipe(pd.to_datetime)
    frame["cand_days_since_opened"] = (
        pd.to_datetime(frame["TARGET_DATE"]) - opened
    ).dt.days.astype(float)
    frame["cand_has_closed_date"] = _static_map(frame, "CLOSED_ON").notna().astype(float)
    frame["cand_open_hours_per_week"] = frame["PLACEKEY"].map(_open_hours_per_week)
    frame["cand_is_open_today"] = [
        _open_today_lookup.get(pk, {}).get(dow, np.nan)
        for pk, dow in zip(frame["PLACEKEY"], frame["target_day_of_week"].astype(int))
    ]
    return [
        "cand_days_since_opened", "cand_has_closed_date",
        "cand_open_hours_per_week", "cand_is_open_today",
    ], []
    # cand_days_since_opened is NaN for most PLACEKEYs -- OPENED_ON is only
    # 12.8% populated repo-wide (see VisitorTimeSeriesPrediction.ipynb Step 2.5).


NAMED_HOLIDAY_RULES = {
    "cand_is_new_years_day": lambda d: (d.dt.month == 1) & (d.dt.day == 1),
    "cand_is_july_4th": lambda d: (d.dt.month == 7) & (d.dt.day == 4),
    "cand_is_christmas_eve": lambda d: (d.dt.month == 12) & (d.dt.day == 24),
    "cand_is_new_years_eve": lambda d: (d.dt.month == 12) & (d.dt.day == 31),
}


def group_calendar_extended(frame):
    d = pd.to_datetime(frame["TARGET_DATE"])
    holidays = pd.DatetimeIndex(sorted(USFederalHolidayCalendar().holidays(
        start=d.min() - pd.Timedelta(days=30), end=d.max() + pd.Timedelta(days=30)
    )))
    hol_arr = holidays.values.astype("datetime64[D]")
    d_arr = d.values.astype("datetime64[D]")
    idx_after = np.searchsorted(hol_arr, d_arr, side="left")
    days_until = np.full(len(d_arr), np.nan)
    days_since = np.full(len(d_arr), np.nan)
    valid_after = idx_after < len(hol_arr)
    days_until[valid_after] = (
        hol_arr[idx_after[valid_after]] - d_arr[valid_after]
    ).astype("timedelta64[D]").astype(float)
    valid_before = idx_after > 0
    days_since[valid_before] = (
        d_arr[valid_before] - hol_arr[idx_after[valid_before] - 1]
    ).astype("timedelta64[D]").astype(float)
    frame["cand_days_until_next_holiday"] = days_until
    frame["cand_days_since_last_holiday"] = days_since
    for name, fn in NAMED_HOLIDAY_RULES.items():
        frame[name] = fn(d).to_numpy().astype(float)
    return [
        "cand_days_until_next_holiday", "cand_days_since_last_holiday",
        *NAMED_HOLIDAY_RULES.keys(),
    ], []


def haversine_km(lat1, lon1, lat2, lon2):
    r = 6371.0
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlmb = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dlmb / 2) ** 2
    return 2 * r * np.arcsin(np.sqrt(a))


_spatial = candidate_static.reset_index()[["PLACEKEY", "MARKET_VISIT", "LATITUDE", "LONGITUDE"]].dropna(
    subset=["LATITUDE", "LONGITUDE"]
).reset_index(drop=True)
_lat, _lon = _spatial["LATITUDE"].to_numpy(dtype=float), _spatial["LONGITUDE"].to_numpy(dtype=float)
_nearby_count = np.zeros(len(_spatial))
_dist_to_center = np.zeros(len(_spatial))
for _, idx in _spatial.groupby("MARKET_VISIT").groups.items():
    idx = np.array(idx)
    mlat, mlon = _lat[idx], _lon[idx]
    center_lat, center_lon = mlat.mean(), mlon.mean()
    _dist_to_center[idx] = haversine_km(mlat, mlon, center_lat, center_lon)
    for i in idx:
        _nearby_count[i] = (haversine_km(_lat[i], _lon[i], mlat, mlon) <= 5.0).sum() - 1
_spatial["cand_nearby_poi_count_5km"] = _nearby_count
_spatial["cand_distance_to_market_center_km"] = _dist_to_center
_spatial_lut = _spatial.set_index("PLACEKEY")


def group_spatial(frame):
    frame["cand_nearby_poi_count_5km"] = frame["PLACEKEY"].map(_spatial_lut["cand_nearby_poi_count_5km"])
    frame["cand_distance_to_market_center_km"] = frame["PLACEKEY"].map(
        _spatial_lut["cand_distance_to_market_center_km"]
    )
    return ["cand_nearby_poi_count_5km", "cand_distance_to_market_center_km"], []


def group_historical_spending(frame):
    merged = frame[["PLACEKEY", "ORIGIN_DATE"]].merge(
        spend_by_key, on=["PLACEKEY", "ORIGIN_DATE"], how="left"
    )
    for c in SPEND_NUMERIC_COLUMNS:
        frame[c] = merged[c].to_numpy()
    return SPEND_NUMERIC_COLUMNS, []


ADD_ONE_IN_GROUPS = {
    "geography": group_geography,
    "physical_poi": group_physical_poi,
    "location_structure": group_location_structure,
    "operating_status": group_operating_status,
    "calendar_extended": group_calendar_extended,
    "spatial_neighborhood": group_spatial,
    "historical_spending": group_historical_spending,
}

In [11]:
add_one_in_train = pd.concat([frame_2021, frame_2022], ignore_index=True)
add_one_in_val = frame_2023.copy()

add_one_in_rows = [
    {"group": "baseline (current FEATURE_COLUMNS)", "val_MAE": full_val_mae, "delta_pct": 0.0, "accepted": None}
]
for group_name, builder in ADD_ONE_IN_GROUPS.items():
    added_numeric, added_categorical = builder(add_one_in_train)
    builder(add_one_in_val)
    mae = _ablation_fit_predict_mae(
        add_one_in_train, add_one_in_val,
        CATEGORICAL_COLUMNS + added_categorical + NUMERIC_COLUMNS + added_numeric,
        CATEGORICAL_COLUMNS + added_categorical,
    )
    delta_pct = (full_val_mae - mae) / full_val_mae * 100
    accepted = delta_pct >= ABLATION_ACCEPT_THRESHOLD * 100
    print(f"+ {group_name}: val_MAE={mae:.4f}  delta={delta_pct:+.2f}%  -> {'ACCEPT' if accepted else 'reject'}")
    add_one_in_rows.append({
        "group": f"+ {group_name}", "val_MAE": mae, "delta_pct": delta_pct, "accepted": accepted,
    })

add_one_in_summary = pd.DataFrame(add_one_in_rows)
display(add_one_in_summary)

del add_one_in_train, add_one_in_val
gc.collect()

+ geography: val_MAE=331.4770  delta=-0.15%  -> reject


+ physical_poi: val_MAE=334.6374  delta=-1.10%  -> reject


+ location_structure: val_MAE=335.0057  delta=-1.21%  -> reject


+ operating_status: val_MAE=338.9707  delta=-2.41%  -> reject


+ calendar_extended: val_MAE=331.9997  delta=-0.31%  -> reject


+ spatial_neighborhood: val_MAE=331.0662  delta=-0.02%  -> reject


+ historical_spending: val_MAE=335.3785  delta=-1.33%  -> reject


,group,val_MAE,delta_pct,accepted
0,baseline (current FEATURE_COLUMNS),330.984846,0.000000,None
1,+ geography,331.476960,-0.148682,False
2,+ physical_poi,334.637438,-1.103553,False
3,+ location_structure,335.005661,-1.214803,False
4,+ operating_status,338.970735,-2.412766,False
5,+ calendar_extended,331.999715,-0.306621,False
6,+ spatial_neighborhood,331.066154,-0.024566,False
7,+ historical_spending,335.378509,-1.327452,False


119

## Hyperparameter tuning (Optuna Bayesian search, top 5 models)

Optuna's TPE sampler models which regions of the search space look promising from trials already
run and samples the next trial accordingly — more efficient per trial than pure random search at
the same budget. Still fit on the same 2021–2022 train / 2023 validation split used throughout
this notebook, and still only tuning the direct model — the recursive/hybrid legs above keep
their own separate hyperparameters untouched.

Rather than adopting a single best trial (which risks overfitting to the one 2023 validation
split), the top 5 trials by validation MAE are kept and averaged together as an ensemble.
`fit_and_predict()` is redefined below to fit all 5 configurations and return the mean of their
predictions — this supersedes `direct_model()`'s role in the original `fit_and_predict()` above,
and cascades to the untouched-2024 test and the final 2025 forecast further down, since both
still call `fit_and_predict()`.

In [12]:
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)

N_TRIALS = 20
TUNING_SEED = 0
TOP_K = 5

tuning_train = pd.concat([frame_2021, frame_2022], ignore_index=True)
X_tuning_train, X_tuning_val, _ = model_frames(tuning_train, frame_2023)
y_tuning_train = np.log1p(tuning_train["y"].to_numpy(dtype=float))


def _params_from_trial(trial: optuna.Trial) -> dict:
    return {
        "num_leaves": trial.suggest_int("num_leaves", 31, 255, log=True),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 300, 1200),
        "min_child_samples": trial.suggest_int("min_child_samples", 20, 400, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
    }


def objective(trial: optuna.Trial) -> float:
    params = _params_from_trial(trial)
    model = lgb.LGBMRegressor(
        objective="regression_l1", random_state=0, verbosity=-1, n_jobs=-1, **params,
    )
    model.fit(X_tuning_train, y_tuning_train)
    prediction = np.clip(np.expm1(model.predict(X_tuning_val)), 0, None)
    return score(frame_2023["y"], prediction)["MAE"]


study = optuna.create_study(
    direction="minimize", sampler=optuna.samplers.TPESampler(seed=TUNING_SEED),
)
study.optimize(objective, n_trials=N_TRIALS)

tuning_results = study.trials_dataframe(attrs=("number", "value", "params", "duration"))
tuning_results = tuning_results.rename(columns={"value": "val_MAE"}).sort_values("val_MAE").reset_index(drop=True)
display(tuning_results.head(10))

print(f"\nBaseline (current direct_model): MAE={full_val_mae:.4f}")
print(f"Best Optuna trial: MAE={tuning_results['val_MAE'].iloc[0]:.4f}")

/Users/rusli/Documents/GitHub/Rice-To-Meet-You/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,number,val_MAE,params_colsample_bytree,params_learning_rate,params_min_child_samples,params_n_estimators,params_num_leaves,params_reg_alpha,params_reg_lambda,params_subsample,duration
0,8,322.623038,0.638439,0.023374,26,1039,46,8.050778,0.074921,0.935178,0 days 00:00:56.233079
1,6,324.177646,0.640818,0.023105,73,813,60,0.006847,0.004418,0.995350,0 days 00:00:53.160260
2,7,324.214074,0.644150,0.017918,41,720,122,0.422006,0.003571,0.663588,0 days 00:01:02.796071
3,9,324.253320,0.648079,0.040257,22,966,243,0.015295,0.002985,0.713123,0 days 00:01:41.765247
4,19,327.751304,0.721115,0.030039,35,1083,111,0.186782,0.104441,0.782558,0 days 00:01:20.065724
5,18,329.728450,0.601431,0.018982,65,859,72,0.031231,0.001016,0.961179,0 days 00:00:53.809339
6,12,329.838880,0.708234,0.023952,49,1133,56,0.277112,0.030767,0.926877,0 days 00:01:02.415375
7,16,330.249880,0.667433,0.020710,29,1095,88,0.416718,0.203125,0.905831,0 days 00:01:15.151052
8,10,331.467501,0.721541,0.010979,183,1158,155,2.589138,0.063878,0.611089,0 days 00:01:52.296630
9,15,332.743242,0.605018,0.029995,69,843,31,9.326746,0.011763,0.962128,0 days 00:00:38.544089



Baseline (current direct_model): MAE=330.9848
Best Optuna trial: MAE=322.6230


In [13]:
TOP_K_PARAMS = [
    {k[len("params_"):]: v for k, v in row.items() if k.startswith("params_")}
    for _, row in tuning_results.head(TOP_K).iterrows()
]
print(f"Top {TOP_K} trials (by val_MAE):")
for i, p in enumerate(TOP_K_PARAMS):
    print(f"  #{i}: MAE={tuning_results['val_MAE'].iloc[i]:.4f}  {p}")


def fit_and_predict(train: pd.DataFrame, test: pd.DataFrame):
    """Top 5 Optuna ensemble: fits one model per TOP_K_PARAMS config and
    averages their predictions. Supersedes the single-model version above --
    only the untouched-2024 test and 2025 forecast below pick this up, since
    validation_pred/full_val_mae above were already computed with the
    original single hand-picked direct_model()."""
    X_train, X_test, categories = model_frames(train, test)
    y_train = np.log1p(train["y"].to_numpy(dtype=float))
    models, predictions = [], []
    for params in TOP_K_PARAMS:
        model = lgb.LGBMRegressor(
            objective="regression_l1", random_state=0, verbosity=-1, n_jobs=-1, **params,
        )
        model.fit(X_train, y_train)
        predictions.append(np.clip(np.expm1(model.predict(X_test)), 0, None))
        models.append(model)
    ensemble_prediction = np.mean(predictions, axis=0)
    return models, ensemble_prediction, categories


_, ensemble_val_pred, _ = fit_and_predict(tuning_train, frame_2023)
ensemble_val_mae = score(frame_2023["y"], ensemble_val_pred)["MAE"]
print(
    f"\nTop-{TOP_K} ensemble on 2023 validation: MAE={ensemble_val_mae:.4f}  "
    f"(baseline {full_val_mae:.4f}, single-best trial {tuning_results['val_MAE'].iloc[0]:.4f})"
)

del tuning_train, X_tuning_train, X_tuning_val, y_tuning_train
gc.collect()

Top 5 trials (by val_MAE):
  #0: MAE=322.6230  {'colsample_bytree': 0.6384393631575852, 'learning_rate': 0.023373576487028164, 'min_child_samples': 26, 'n_estimators': 1039, 'num_leaves': 46, 'reg_alpha': 8.050778165087026, 'reg_lambda': 0.07492121411354245, 'subsample': 0.9351779629995216}
  #1: MAE=324.1776  {'colsample_bytree': 0.6408179242992113, 'learning_rate': 0.023105255265581114, 'min_child_samples': 73, 'n_estimators': 813, 'num_leaves': 60, 'reg_alpha': 0.006847105576684045, 'reg_lambda': 0.004418125737902547, 'subsample': 0.9953495352236905}
  #2: MAE=324.2141  {'colsample_bytree': 0.644150056465722, 'learning_rate': 0.017918085415441008, 'min_child_samples': 41, 'n_estimators': 720, 'num_leaves': 122, 'reg_alpha': 0.42200573971950167, 'reg_lambda': 0.003570522756718293, 'subsample': 0.6635878334582078}
  #3: MAE=324.2533  {'colsample_bytree': 0.6480786244852675, 'learning_rate': 0.04025738117667019, 'min_child_samples': 22, 'n_estimators': 966, 'num_leaves': 243, 'reg_alph


Top-5 ensemble on 2023 validation: MAE=316.6154  (baseline 330.9848, single-best trial 322.6230)


161

In [14]:
# Final test: this is the first and only use of actual 2024 labels for scoring.
frame_2024 = make_direct_frame("2023-12-31")
pretest_train = pd.concat(
    [frame_2021, frame_2022, frame_2023], ignore_index=True
)
test_model, test_pred, _ = fit_and_predict(pretest_train, frame_2024)

print("Fitting recursive model and rolling it forward through 2024...")
test_recursive_model, test_recursive_categories = fit_recursive_model("2023-12-31")
test_recursive_df = recursive_placekey_forecast(
    "2023-12-31", pd.date_range("2024-01-01", "2024-12-31", freq="D"),
    test_recursive_model, test_recursive_categories,
)
test_recursive_eval = frame_2024.merge(
    test_recursive_df, on=["PLACEKEY", "TARGET_DATE"], how="inner"
)
test_hybrid_pred = hybrid_predictions(frame_2024, test_pred, test_recursive_df)

test_results = pd.DataFrame([
    {
        "split": "2024 pooled-direct model",
        **score(frame_2024["y"], test_pred),
    },
    {
        "split": "2024 recursive model",
        **score(test_recursive_eval["y"], test_recursive_eval["recursive_pred"]),
    },
    {
        "split": f"2024 hybrid model (recursive <= {HYBRID_THRESHOLD_DAYS}d, direct after)",
        **score(frame_2024["y"], test_hybrid_pred),
    },
    {
        "split": "2024 lag364 baseline",
        **score(frame_2024["y"], frame_2024["lag364_baseline"]),
    },
])
display(test_results)

by_horizon = pd.DataFrame({
    "horizon_days": frame_2024["horizon_days"].astype(int),
    "absolute_error": np.abs(frame_2024["y"].to_numpy() - test_pred),
})
by_horizon["horizon_bucket"] = pd.cut(
    by_horizon["horizon_days"],
    bins=[0, 31, 90, 180, 270, 366],
    labels=["1-31", "32-90", "91-180", "181-270", "271-366"],
)
display(
    by_horizon.groupby("horizon_bucket", observed=True)["absolute_error"]
    .mean().rename("MAE").reset_index()
)

del test_model, test_pred, pretest_train, by_horizon
del test_recursive_model, test_recursive_df, test_recursive_eval, test_hybrid_pred
gc.collect()

Fitting recursive model and rolling it forward through 2024...


,split,MAE,RMSE,sMAPE,R2
0,2024 pooled-direct model,319.014522,879.462471,21.172635,0.962983
1,2024 recursive model,536.090938,1255.170208,42.435346,0.924601
2,"2024 hybrid model (recursive <= 90d, direct af...",350.040056,924.645124,23.149598,0.959082
3,2024 lag364 baseline,353.989510,1106.118199,14.911037,0.941445


,horizon_bucket,MAE
0,1-31,341.998622
1,32-90,263.975643
2,91-180,268.477896
3,181-270,295.368330
4,271-366,414.964943


0

## Refit through 2024 and predict 2025 directly

Each output row is independently predicted from the 2024-12-31 origin,
historical analog dates, static attributes at the origin, and its own target
calendar. The following assertions verify that there is exactly one row for
each eligible `PLACEKEY × 2025 date` and no reference date exceeds the
production origin.

In [15]:
final_train = pd.concat(
    [frame_2021, frame_2022, frame_2023, frame_2024], ignore_index=True
)
future_2025 = make_direct_frame(
    FORECAST_ORIGIN,
    target_dates=FORECAST_DATES,
    include_target=False,
)

final_model, prediction_2025, _ = fit_and_predict(final_train, future_2025)
predictions_2025 = future_2025[["PLACEKEY", "TARGET_DATE"]].rename(
    columns={"TARGET_DATE": "LOCAL_DATE"}
)
predictions_2025[f"predicted_{TARGET}"] = prediction_2025

expected_rows = (
    future_2025["PLACEKEY"].nunique() * len(FORECAST_DATES)
)
assert len(predictions_2025) == expected_rows
assert not predictions_2025.duplicated(["PLACEKEY", "LOCAL_DATE"]).any()
assert predictions_2025["LOCAL_DATE"].min() == FORECAST_DATES.min()
assert predictions_2025["LOCAL_DATE"].max() == FORECAST_DATES.max()
assert all(
    (future_2025[f"reference_date_{years}y"] <= FORECAST_ORIGIN).all()
    for years in (1, 2, 3)
)

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
predictions_2025.to_csv(OUTPUT_PATH, index=False)
print(
    f"Generated {len(predictions_2025):,} independent direct predictions "
    f"for {predictions_2025['PLACEKEY'].nunique():,} PLACEKEYs"
)
print(f"Wrote -> {OUTPUT_PATH}")
display(predictions_2025.head())

Generated 1,270,930 independent direct predictions for 3,482 PLACEKEYs
Wrote -> /Users/rusli/Documents/GitHub/Rice-To-Meet-You/visitor_data/output/visitor_predictions_2025_direct.csv


,PLACEKEY,LOCAL_DATE,predicted_AVERAGE_DAILY_VISITS
0,222-222@5pr-2xs-xqz,2025-01-01,6.046882
1,222-222@5pr-2xs-xqz,2025-01-02,9.393802
2,222-222@5pr-2xs-xqz,2025-01-03,9.293914
3,222-222@5pr-2xs-xqz,2025-01-04,0.629592
4,222-222@5pr-2xs-xqz,2025-01-05,0.086795


## Leakage audit

- No feature uses a target from inside the forecast year.
- All historical reference dates are on or before the fixed origin.
- The 2024 test reproduces the exact 365/366-day production horizon.
- Model selection does not use the 2024 test.
- The production fit is explicitly bounded by `2024-12-31`.
- Calendar-year joins replace row-based 365-day shifts around leap years.
- Predictions are never recursively reused, so forecast errors cannot
  compound through lag features.